# Lia — tentativa de LoRA Qwen3-TTS no Colab T4

Notebook **separado** do benchmark e do notebook de fine-tuning completo. Usa o ZIP curado que você já baixou; não pede nem retranscreve o WAV original. A saída é um adapter LoRA pequeno. Dados, checkpoints e áudios gerados ficam no runtime privado Colab.

**Tentativa experimental para T4, sem garantia de caber:** LoRA reduz pesos treináveis/estado do otimizador, mas ainda precisa guardar ativações. Primeiro executa exatamente um passo de treino em FP32 + gradient checkpointing + SDPA. Só inicie a época completa depois de o smoke test passar. Se houver OOM/incompatibilidade, não haverá adapter válido e a inferência fica bloqueada; isso evita confundir áudio do modelo-base com voz da Lia.

O aviso de `flash-attn` ausente é esperado: esta receita usa SDPA do PyTorch; não compila FlashAttention. O wrapper Python `sox` é instalado porque Qwen importa módulos 25 Hz, mas a rota treinada usa tokenizer 12 Hz.

A receita comunitária `instavar/qwen3-tts-lora-finetuning` está pinada e aplicada a um commit Qwen upstream específico, com correções conhecidas de alinhamento de loss e `text_projection`. Não é o trainer oficial nem valida qualidade PT-BR. Os rótulos emotion do ZIP são metadados: o trainer não os condiciona diretamente. A receita comunitária também ignora o campo `language` e treina o prefixo de idioma automático; a inferência deve usar `auto` para manter o mesmo alinhamento. `Portuguese` não distingue Brasil de Portugal. A adaptação só pode aprender PT-BR dos códigos de áudio se o treino realmente concluir.

**Configuração inicial:** Qwen 0.6B, base FP32 (não BF16/FP16), SDPA, batch 1, gradient accumulation 4, rank 4, Q/K/V/O, LR 2e-6, uma época. Os pesos-base não são alterados.

In [ ]:
# 0. Dependências auxiliares isoladas; mantém Torch/Torchaudio CUDA do Colab
from pathlib import Path
import os, sys, subprocess, json, csv, shutil, zipfile, hashlib, importlib.util, gc, time
import torch
ROOT=Path('/content/lia_qwen_lora'); ROOT.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(ROOT/'hf_home')  # definido antes de importar huggingface_hub
DEPS=ROOT/'python_deps'; DEPS.mkdir(exist_ok=True)
packages=['transformers==4.57.3','accelerate==1.12.0','peft==0.17.1','huggingface_hub==0.36.2','safetensors==0.7.0','tokenizers==0.22.1','librosa==0.11.0','soxr==0.5.0.post1','sox==1.5.0','einops==0.8.1','onnxruntime==1.23.2']
marker=DEPS/'.lora_deps_ready'
if not marker.exists():
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--no-deps','--target',str(DEPS),*packages],check=True)
    marker.write_text('\n'.join(packages)+'\n',encoding='utf-8')
sys.path.insert(0,str(DEPS))
# Qwen é áudio; impede auto-detecção de uma torchvision incompatível não usada.
import transformers.utils.import_utils as _hf_import_utils
_hf_import_utils._torchvision_available=False
import transformers, accelerate, peft, huggingface_hub, soundfile as sf
from google.colab import files
from IPython.display import display, Audio
if not torch.cuda.is_available(): raise RuntimeError('Selecione uma GPU CUDA (T4) no runtime Colab.')
props=torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GiB | BF16 reportado: {torch.cuda.is_bf16_supported()}')
print('Torch ativo:',torch.__version__,'| Transformers isolado:',transformers.__version__,'| PEFT:',peft.__version__)
print('Dados/checkpoints em',ROOT)
existing_zips=sorted(p for p in ROOT.glob('*.zip') if p.is_file())
if len(existing_zips)==1:
    bundle=existing_zips[0]
    print('Reutilizando o ZIP curado já presente:',bundle.name)
else:
    print('Selecione o ZIP curado já baixado.')
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Envie exatamente um ZIP.')
    zip_name,zip_bytes=next(iter(uploaded.items()))
    if not zip_name.lower().endswith('.zip'): raise ValueError('Esperado um arquivo ZIP.')
    bundle=ROOT/Path(zip_name).name; bundle.write_bytes(zip_bytes); del zip_bytes,uploaded; gc.collect()
with zipfile.ZipFile(bundle) as zf:
    root_resolved=ROOT.resolve()
    for item in zf.infolist():
        target=(ROOT/item.filename).resolve()
        if target!=root_resolved and root_resolved not in target.parents: raise ValueError('Caminho inseguro no ZIP; extração bloqueada.')
    zf.extractall(ROOT)
manifest=json.loads((ROOT/'training_bundle_manifest.json').read_text(encoding='utf-8'))
if manifest.get('format')!='lia-qwen-training-bundle-v1': raise ValueError('Formato de pacote inesperado.')
for required in ('train_raw.jsonl','holdout_raw.jsonl','review.csv','review_approval.json','emotion_audit.csv','speaker_reference.wav'):
    if not (ROOT/required).is_file(): raise FileNotFoundError(f'Arquivo obrigatório ausente: {required}')
review_hash=hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest()
approval=json.loads((ROOT/'review_approval.json').read_text(encoding='utf-8'))
if approval.get('review_csv_sha256')!=review_hash or manifest.get('review_csv_sha256')!=review_hash: raise RuntimeError('Hash da revisão não corresponde à aprovação; pacote alterado/incompleto.')
actual_train=sum(1 for line in (ROOT/'train_raw.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
actual_holdout=sum(1 for line in (ROOT/'holdout_raw.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
if actual_train!=manifest.get('train_rows') or actual_holdout!=manifest.get('holdout_rows'):
    raise RuntimeError(f'Contagem no manifest diverge do conteúdo: train {actual_train}/{manifest.get("train_rows")}, holdout {actual_holdout}/{manifest.get("holdout_rows")}')
print('Pacote curado:',actual_train,'treino;',actual_holdout,'holdout; review hash',review_hash[:12])

## 1. Validar e limitar duração para a T4
Expande caminhos portáteis/antigos do ZIP para caminhos absolutos locais, verifica os WAVs e limita a 12 s para reduzir memória de ativações. Os originais não são alterados; holdout permanece separado.

In [ ]:
# 1. Paths locais + teto de duração (apenas para esta experiência T4)
from pathlib import Path
import soundfile as sf
MAX_CLIP_SECONDS=12.0

def resolve_bundled_audio(value):
    p=Path(value)
    if p.is_absolute() and p.is_file() and p.is_relative_to(ROOT.resolve()): return p.resolve()
    if not p.is_absolute():
        portable=(ROOT/p).resolve()
        if portable.is_file() and portable.is_relative_to(ROOT.resolve()): return portable
    # JSONL de runtime antigo pode guardar caminho absoluto; resolve pelo nome no ZIP portátil.
    matches=list(ROOT.rglob(p.name))
    matches=[x.resolve() for x in matches if x.is_file() and x.suffix.lower()=='.wav']
    if len(matches)!=1: raise FileNotFoundError(f'Não consegui resolver univocamente o áudio {value!r}: {matches[:4]}')
    return matches[0]

def prepare_split(src_name,dst_name,min_rows):
    kept=[]; dropped=[]; refs=set()
    for line_no,line in enumerate((ROOT/src_name).read_text(encoding='utf-8').splitlines(),1):
        if not line.strip(): continue
        row=json.loads(line); audio=resolve_bundled_audio(row['audio']); ref=resolve_bundled_audio(row['ref_audio'])
        if not audio.is_relative_to(ROOT.resolve()) or not ref.is_relative_to(ROOT.resolve()): raise ValueError('Áudio fora do ZIP/runtime privado.')
        info=sf.info(audio)
        if info.channels!=1 or info.samplerate!=24000: raise ValueError(f'{audio.name}: esperava mono 24 kHz, recebi {info.channels}ch/{info.samplerate}Hz')
        refs.add(ref)
        if info.duration>MAX_CLIP_SECONDS: dropped.append((audio.name,info.duration)); continue
        row['audio']=str(audio); row['ref_audio']=str(ref); kept.append(row)
    if len(refs)!=1: raise ValueError(f'{src_name}: esperado um ref_audio fixo, encontrei {len(refs)}.')
    ref_info=sf.info(next(iter(refs)))
    if ref_info.channels!=1 or ref_info.samplerate!=24000: raise ValueError(f'ref_audio: esperava mono 24 kHz, recebi {ref_info.channels}ch/{ref_info.samplerate}Hz')
    if len(kept)<min_rows: raise ValueError(f'{src_name}: restaram {len(kept)} recortes <= {MAX_CLIP_SECONDS}s; mínimo para esta divisão: {min_rows}.')
    (ROOT/dst_name).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in kept),encoding='utf-8')
    return kept,dropped
train_rows,dropped_train=prepare_split('train_raw.jsonl','train_t4.jsonl',5)
val_rows,dropped_val=prepare_split('holdout_raw.jsonl','holdout_t4.jsonl',1)
print(f'Treino: {len(train_rows)} recortes, {sum(sf.info(r["audio"]).duration for r in train_rows)/60:.1f} min; holdout: {len(val_rows)}.')
print(f'Excluídos temporariamente por >{MAX_CLIP_SECONDS}s: treino {len(dropped_train)}, holdout {len(dropped_val)}.')
print('Emoções anotadas (metadado, não condicionamento direto):',sorted({r.get('emotion_audit','nao_rotulado') for r in train_rows}))

## 2. Pin Qwen + correções LoRA e preparar modelo
O patch comunitário se aplica ao commit upstream fixo `0c6a7c…`; o notebook verifica a revisão e o `git apply --check`. Ajustes estritos para T4: FP32, sem TensorBoard, gradient checkpointing, batch do tokenizer 1 e SDPA. Se o código upstream divergir, a célula para em vez de fazer patch aproximado.

In [ ]:
# 2. Repositórios e snapshots de modelo pinados
QWEN_COMMIT='0c6a7cbb6c8421a46332f8c2434c7825c4c855ef'
COMPANION_COMMIT='079db4ba844037f970b488dc96b32e5c7490642f'
MODEL_ID='Qwen/Qwen3-TTS-12Hz-0.6B-Base'; MODEL_REV='5d83992436eae1d760afd27aff78a71d676296fc'
TOKENIZER_ID='Qwen/Qwen3-TTS-Tokenizer-12Hz'; TOKENIZER_REV='7dd38ad4e9bad454aae9cd937d0cd577604fe229'
QWEN_DIR=ROOT/'Qwen3-TTS'; COMP_DIR=ROOT/'qwen3-tts-lora-finetuning'
def clone_pinned(url,dest,sha):
    if not dest.exists(): subprocess.run(['git','clone','--quiet','--no-checkout',url,str(dest)],check=True)
    subprocess.run(['git','-C',str(dest),'fetch','--quiet','--depth','1','origin',sha],check=True)
    # Idempotente: remove o patch local da execução anterior antes de reaplicá-lo.
    subprocess.run(['git','-C',str(dest),'reset','--hard',sha],check=True,stdout=subprocess.DEVNULL)
    subprocess.run(['git','-C',str(dest),'clean','-fd'],check=True,stdout=subprocess.DEVNULL)
    got=subprocess.check_output(['git','-C',str(dest),'rev-parse','HEAD'],text=True).strip()
    if got!=sha: raise RuntimeError(f'Commit inesperado: {got}')
clone_pinned('https://github.com/QwenLM/Qwen3-TTS.git',QWEN_DIR,QWEN_COMMIT)
clone_pinned('https://github.com/instavar/qwen3-tts-lora-finetuning.git',COMP_DIR,COMPANION_COMMIT)
patch=COMP_DIR/'patches'/'0001-qwen3-tts-lora.patch'
subprocess.run(['git','-C',str(QWEN_DIR),'apply','--check',str(patch)],check=True)
subprocess.run(['git','-C',str(QWEN_DIR),'apply',str(patch)],check=True)
trainer=QWEN_DIR/'finetuning'/'sft_12hz_lora.py'
source=trainer.read_text(encoding='utf-8')
def replace_once(text,old,new,label):
    n=text.count(old)
    if n!=1: raise RuntimeError(f'Patch T4 abortado: {label} devia ocorrer uma vez, ocorreu {n}.')
    return text.replace(old,new,1)
source=replace_once(source,'torch_dtype=torch.bfloat16,','dtype=torch.float32,','FP32 para T4; API Transformers atual')
source=replace_once(source,'log_with="tensorboard",','log_with=None,','sem TensorBoard')
source=replace_once(source,'    config = AutoConfig.from_pretrained(MODEL_PATH)\n','''    config = AutoConfig.from_pretrained(MODEL_PATH)
    qwen3tts.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    # Desliga KV cache em todos os submodelos; cache não é necessário no SFT e custa VRAM.
    for _module in qwen3tts.model.modules():
        _config = getattr(_module, "config", None)
        if _config is not None and hasattr(_config, "use_cache"):
            _config.use_cache = False
    # Qwen3TTS é multimodal e não implementa o hook genérico do Transformers.
    # PEFT 0.17 chama enable_input_require_grads() ao detectar checkpointing;
    # exponha o embedding textual usado pelo trainer para esse hook funcionar.
    qwen3tts.model.get_input_embeddings = lambda: qwen3tts.model.talker.model.text_embedding
''','gradient checkpointing')
source=replace_once(source,'    parser.add_argument("--eval_every", type=int, default=1)\n','''    parser.add_argument("--eval_every", type=int, default=1)
    parser.add_argument("--max_train_steps", type=int, default=0)
''','limite do smoke')
source=replace_once(source,'            if step % 10 == 0:\n                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")\n','''            if step % 10 == 0:
                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")
            if args.max_train_steps and step + 1 >= args.max_train_steps:
                peak=torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.0
                accelerator.print(f"Smoke test encerrou em {step+1} passo; pico CUDA alocado {peak:.2f} GiB; nenhum checkpoint salvo.")
                return
''','saída limitada do smoke')
trainer.write_text(source,encoding='utf-8')
prep=QWEN_DIR/'finetuning'/'prepare_data.py'; ps=prep.read_text(encoding='utf-8')
if ps.count('BATCH_INFER_NUM = 32')!=1: raise RuntimeError('Batch do tokenizer mudou; revisão necessária.')
prep.write_text(ps.replace('BATCH_INFER_NUM = 32','BATCH_INFER_NUM = 1',1),encoding='utf-8')
infer=QWEN_DIR/'finetuning'/'infer_lora_custom_voice.py'; ins=infer.read_text(encoding='utf-8')
ins=replace_once(ins,'if __name__ == \"__main__\":\n','if __name__ == \"__main__\":\n    main()\n','entrypoint do helper de inferência')
ins=replace_once(ins,'torch_dtype=dtype_map[args.dtype],','dtype=dtype_map[args.dtype],','API Transformers atual na inferência')
ins=replace_once(ins,'    parser.add_argument("--language", default="auto")\n','''    parser.add_argument("--language", default="auto")
    parser.add_argument("--instruct", default="")
''','instruct CLI')
ins=replace_once(ins,'        language=args.language,\n        max_new_tokens=args.max_new_tokens,\n','''        language=args.language,
        instruct=args.instruct or None,
        max_new_tokens=args.max_new_tokens,
''','passagem do instruct')
infer.write_text(ins,encoding='utf-8')
os.environ['HF_HOME']=str(ROOT/'hf_home')
from huggingface_hub import snapshot_download
if globals().get('model_path') and Path(model_path).is_dir():
    print('Reutilizando o snapshot local do modelo base:',model_path)
else:
    model_path=snapshot_download(MODEL_ID,revision=MODEL_REV,cache_dir=str(ROOT/'hf_home'))
if globals().get('tokenizer_path') and Path(tokenizer_path).is_dir():
    print('Reutilizando o snapshot local do tokenizer:',tokenizer_path)
else:
    tokenizer_path=snapshot_download(TOKENIZER_ID,revision=TOKENIZER_REV,cache_dir=str(ROOT/'hf_home'))
train_env=os.environ.copy(); train_env['HF_HOME']=str(ROOT/'hf_home')
(DEPS/'sitecustomize.py').write_text('try:\n import transformers.utils.import_utils as _u\n _u._torchvision_available=False\nexcept Exception:\n pass\n',encoding='utf-8')
train_env['PYTHONPATH']=os.pathsep.join([str(DEPS),str(QWEN_DIR),train_env.get('PYTHONPATH','')])
train_env['PYTHONUNBUFFERED']='1'
print('Patch fixo aplicado. Modelos baixados; trainer: FP32, SDPA, rank 4, batch 1, gradient checkpointing.')

## 3. Tokenizar os códigos de áudio
Executa o tokenizer oficial sobre treino e holdout. Batch 1 para limitar a VRAM. Ainda não treina.

In [ ]:
# 3. Preparar códigos de áudio, mostrando o traceback real do subprocesso
import collections
# Primeiro valida imports/dependências sem carregar pesos; se falhar, imprime toda a mensagem.
# Qwen importa também o wrapper Python 'sox' durante o carregamento dos módulos 25 Hz.
# A rota Lia usa tokenizer 12 Hz; não é necessário compilar FlashAttention nem instalar o binário SoX.
if importlib.util.find_spec('sox') is None:
    print('Instalando o wrapper Python sox==1.5.0 no target isolado do notebook...')
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--no-deps','--target',str(DEPS),'sox==1.5.0'],check=True)
    importlib.invalidate_caches()
    sys.path.insert(0,str(DEPS))
probe_code="import torch,torchaudio,transformers,accelerate,peft,librosa,soundfile,sox; from qwen_tts import Qwen3TTSTokenizer; print('imports OK',torch.__version__,torchaudio.__version__,transformers.__version__,peft.__version__)"
probe=subprocess.run([sys.executable,'-c',probe_code],env=train_env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(probe.stdout or '(sem saída do teste de imports)')
if probe.returncode: raise RuntimeError(f'Teste de imports falhou (exit {probe.returncode}); veja o traceback acima.')

prep=QWEN_DIR/'finetuning'/'prepare_data.py'
for src,dst in [('train_t4.jsonl','train_t4_codes.jsonl'),('holdout_t4.jsonl','holdout_t4_codes.jsonl')]:
    outpath=ROOT/dst
    input_path=ROOT/src
    expected=sum(1 for line in input_path.read_text(encoding='utf-8').splitlines() if line.strip())
    if outpath.is_file():
        try:
            with outpath.open(encoding='utf-8') as f: cached=[json.loads(line) for line in f if line.strip()]
            reusable=(len(cached)==expected and all('audio_codes' in row for row in cached))
        except Exception: reusable=False
        if reusable:
            print(f'Reutilizando {dst} validado ({expected} exemplos).'); continue
        outpath.unlink()  # remove somente saída gerada inválida/incompleta
    cmd=[sys.executable,'-u',str(prep),'--device','cuda:0','--tokenizer_model_path',tokenizer_path,
         '--input_jsonl',str(input_path),'--output_jsonl',str(outpath)]
    print('\nExecutando:',src,'→',dst,flush=True)
    proc=subprocess.Popen(cmd,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    tail=collections.deque(maxlen=100)
    for line in proc.stdout:
        print(line,end='',flush=True); tail.append(line)
    rc=proc.wait()
    if rc:
        print('\n--- últimas linhas do processo de tokenização ---\n'+''.join(tail),flush=True)
        raise RuntimeError(f'prepare_data.py falhou no split {src} (exit {rc}); traceback integral exibido acima.')
    if not outpath.is_file() or outpath.stat().st_size==0: raise RuntimeError(f'Preparação incompleta: {dst}')
    with outpath.open(encoding='utf-8') as f: coded=[json.loads(line) for line in f if line.strip()]
    expected=sum(1 for line in (ROOT/src).read_text(encoding='utf-8').splitlines() if line.strip())
    if len(coded)!=expected or any('audio_codes' not in row for row in coded):
        raise RuntimeError(f'{dst}: códigos ausentes ou contagem divergente ({len(coded)}/{expected}); não prossiga ao smoke.')
print('JSONLs codificados localmente; nenhum áudio enviado a serviço.')

## 4. Smoke test: exatamente um passo
Carrega o modelo, executa exatamente um forward/backward e um update real do adapter (acumulação 1), depois encerra sem salvar checkpoint. Se OOM/incompatibilidade, a célula levanta erro e o treino longo não começa.

In [ ]:
# 4. Um passo descartável
status_path=ROOT/'lora_training_success.json'
status_path.unlink(missing_ok=True)  # esta tentativa invalida provenance antiga até um novo treino passar
training_completed=False
BASE_ARGS=[sys.executable,'-u',str(trainer),'--init_model_path',model_path,'--output_model_path',str(ROOT/'smoke_discarded'),
 '--train_jsonl',str(ROOT/'train_t4_codes.jsonl'),'--batch_size','1','--lr','2e-6','--num_epochs','1','--speaker_name','lia_lora',
 '--gradient_accumulation_steps','1','--mixed_precision','no','--attn_implementation','sdpa','--lora_rank','4','--lora_alpha','8',
 '--lora_dropout','0.05','--lora_target_modules','q_proj,k_proj,v_proj,o_proj','--max_train_steps','1']
print('Smoke test na T4; se falhar, pare aqui.',flush=True)
proc=subprocess.Popen(BASE_ARGS,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
rc=proc.wait()
if rc: raise RuntimeError(f'Smoke test LoRA falhou (exit {rc}); treino completo NÃO iniciado.')
print('Smoke test passou. A próxima célula inicia uma época experimental.')

## 5. Treinar adapter (uma época inicial)
Experimento curto, não modelo final. Loss e holdout não avaliam expressividade nem sotaque. Adapter, configuração e embedding de speaker são salvos sem tocar na base.

In [ ]:
# 5. Executar uma época após smoke bem-sucedido
TRAIN_LORA=True
if TRAIN_LORA:
    run_id=time.strftime('%Y%m%d_%H%M%S')+'_'+str(time.time_ns()%1_000_000_000)
    adapter_dir=ROOT/f'lora_adapter_{run_id}'  # novo diretório: não reutiliza checkpoint velho
    status_path=ROOT/'lora_training_success.json'
    status_path.unlink(missing_ok=True)  # invalida qualquer status antigo antes da nova tentativa
    cmd=[sys.executable,'-u',str(trainer),'--init_model_path',model_path,'--output_model_path',str(adapter_dir),
      '--train_jsonl',str(ROOT/'train_t4_codes.jsonl'),'--val_jsonl',str(ROOT/'holdout_t4_codes.jsonl'),
      '--batch_size','1','--lr','2e-6','--num_epochs','1','--speaker_name','lia_lora','--gradient_accumulation_steps','4',
      '--mixed_precision','no','--attn_implementation','sdpa','--lora_rank','4','--lora_alpha','8','--lora_dropout','0.05',
      '--lora_target_modules','q_proj,k_proj,v_proj,o_proj']
    proc=subprocess.Popen(cmd,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout: print(line,end='',flush=True)
    rc=proc.wait()
    if rc: raise RuntimeError(f'Treino falhou (exit {rc}); veja log acima.')
    checkpoints=sorted(adapter_dir.glob('checkpoint-epoch-*'))
    if not checkpoints: raise FileNotFoundError('Treino terminou sem checkpoint adapter.')
    adapter_path=checkpoints[-1]
    required_artifacts=['adapter_config.json','adapter_model.safetensors','speaker_embedding.safetensors']
    missing=[name for name in required_artifacts if not (adapter_path/name).is_file()]
    if missing: raise FileNotFoundError(f'Checkpoint não é adapter LoRA completo; faltam {missing}. Inferência bloqueada.')
    def sha256_file(path):
        digest=hashlib.sha256()
        with Path(path).open('rb') as f:
            for block in iter(lambda:f.read(1024*1024),b''): digest.update(block)
        return digest.hexdigest()
    status={'status':'success','adapter_path':str(adapter_path),'base_model_id':MODEL_ID,'base_revision':MODEL_REV,
            'qwen_commit':QWEN_COMMIT,'lora_commit':COMPANION_COMMIT,'train_rows':len(train_rows),
            'train_codes_sha256':sha256_file(ROOT/'train_t4_codes.jsonl'),
            'speaker_reference_sha256':sha256_file(ROOT/'speaker_reference.wav'),
            'created_at':time.strftime('%Y-%m-%dT%H:%M:%S%z')}
    status_path.write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
    training_completed=True
    print('Adapter LoRA validado e registrado:',adapter_path)
else: print('Treino desativado.')

## 6. Comparar instrução alegre e neutra
Duas falas curtas com o mesmo texto, adapter e seed. `instruct` é comando de inferência, não rótulo supervisionado de emoção no trainer. Avalie timbre, naturalidade, PT-BR e diferença emocional.

In [ ]:
# 6. Síntese comparativa
from IPython.display import Audio,display
status_path=ROOT/'lora_training_success.json'
if not status_path.is_file():
    raise RuntimeError('Inferência bloqueada: esta sessão não registrou um treino LoRA concluído. Um áudio-base genérico não serve como resultado da Lia.')
status=json.loads(status_path.read_text(encoding='utf-8'))
if status.get('status')!='success' or status.get('base_revision')!=MODEL_REV or status.get('qwen_commit')!=QWEN_COMMIT:
    raise RuntimeError('Proveniência do adapter inválida/divergente; não vou gerar áudio como se fosse a Lia.')
if status.get('train_codes_sha256')!=sha256_file(ROOT/'train_t4_codes.jsonl'):
    raise RuntimeError('O JSONL de áudio usado no treino difere do atual; adapter recusado.')
if status.get('speaker_reference_sha256')!=sha256_file(ROOT/'speaker_reference.wav'):
    raise RuntimeError('A referência de speaker difere da usada no treino; adapter recusado.')
adapter_path=Path(status['adapter_path']).resolve()
if not adapter_path.is_relative_to(ROOT.resolve()): raise RuntimeError('Caminho do adapter fora do workspace privado.')
if not (adapter_path/'adapter_model.safetensors').is_file() or not (adapter_path/'speaker_embedding.safetensors').is_file():
    raise FileNotFoundError('O adapter registrado está incompleto; inferência bloqueada.')
infer=QWEN_DIR/'finetuning'/'infer_lora_custom_voice.py'
text_emotion='Você conseguiu! Eu sabia que era capaz. Estou muito orgulhosa de você!'
text_dialect='A tia Denise chegou à cidade às oito da noite. Ela deixou o celular no ônibus, mas a gente encontrou depois.'
cases=[
 ('alegre_scale03',text_emotion,'Fale em português brasileiro, com alegria calorosa e entusiasmo genuíno; natural, sem gritar.',0.3),
 ('alegre_scale10',text_emotion,'Fale em português brasileiro, com alegria calorosa e entusiasmo genuíno; natural, sem gritar.',1.0),
 ('regional_scale10',text_dialect,'Português brasileiro natural, sem sotaque de Portugal; pronuncie naturalmente cidade, noite, tia e celular em português do Brasil.',1.0),
]
print(f'Adapter conferido: {adapter_path} | {status["train_rows"]} amostras | hash dos códigos e ref conferidos.')
run_tag=adapter_path.name.replace('lora_adapter_','')
outputs=[]
for label,text,instruct,scale in cases:
    out=ROOT/f'lia_{label}_{run_tag}.wav'
    if out.exists(): out.unlink()  # nunca reproduz WAV antigo se a inferência não o recriar
    cmd=[sys.executable,'-u',str(infer),'--base_model_path',model_path,'--adapter_path',str(adapter_path),'--speaker_name','lia_lora',
      '--speaker_id','3000','--text',text,'--language','auto','--output_wav',str(out),'--device','cuda:0','--dtype','fp32',
      '--attn_implementation','sdpa','--lora_scale',str(scale),'--max_new_tokens','160','--seed','1234','--instruct',instruct]
    subprocess.run(cmd,check=True,env=train_env)
    if not out.is_file() or out.stat().st_size<=44: raise RuntimeError(f'Inferência não criou WAV novo/ válido: {out}')
    info=sf.info(out)
    if info.duration<=0: raise RuntimeError(f'WAV vazio: {out}')
    outputs.append({'case':label,'path':str(out),'seconds':info.duration,'sample_rate':info.samplerate,'lora_scale':scale})
    print(label,info.samplerate,'Hz',f'{info.duration:.2f}s',out)
    display(Audio(filename=str(out)))
(ROOT/'lia_lora_inference_manifest.json').write_text(json.dumps({'status':'generated_from_verified_adapter','adapter_path':str(adapter_path),'base_revision':MODEL_REV,'outputs':outputs},ensure_ascii=False,indent=2),encoding='utf-8')
print('Adapter de origem:',adapter_path,'| idioma de inferência: auto (sem token de dialeto PT-BR/PT-PT).')
print('Adapter de origem:',adapter_path,'| outputs/manifeste privados em',ROOT)

### Limites e próximos passos
- Se o smoke der OOM, a T4 pode não comportar backprop mesmo com LoRA. O ajuste conservador seguinte é teto de áudio 8 s e/ou rank 2 — não forçar precisão instável.
- Se treinar, não substitua o Qwen base automaticamente. Faça avaliação auditiva A/B; loss não mede emoção nem PT-BR.
- Técnica comunitária experimental, não full SFT oficial. Pesos públicos são baixados; ZIP, adapters e WAVs gerados permanecem no Colab. Nenhum serviço hosted recebe a gravação.